# 05 — Same abstract graph, distinct spatial embeddings with KnottedGraph only

This notebook fixes one trivalent theta graph ($V=2,E=3$), constructs several explicit 3D embeddings of that same abstract graph, and uses **KnottedGraph's Yamada polynomial** to test which embeddings it distinguishes.

No Topoly/HOMFLY calculation is used. The knot labels below describe how the embedded edge is constructed from standard parametric knot templates; they are not an independent external certificate.

The logically rigorous conclusion is one-way:

$$
\Upsilon(G_i;A)\neq \Upsilon(G_j;A)
\quad\Longrightarrow\quad
G_i\ \text{and}\ G_j\ \text{are not ambient isotopic}.
$$

Equal Yamada polynomials do **not** imply equivalent embeddings, because the Yamada polynomial is not a complete invariant.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import sympy as sp

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook inside the KnottedGraph checkout.")

import knotted_graph
from knotted_graph.invariants.yamada.native import (
    native_available,
    native_import_error,
)
from knotted_graph.projection import compute_yamada_polynomial

print("Python executable:", sys.executable)
print("Native Yamada backend:", native_available())
print("Native import error:", native_import_error())

A = sp.Symbol("A")
print("KnottedGraph:", Path(knotted_graph.__file__).resolve())


## Experiment controls

Change the values in the next cell to tune the experiment. The parameters are grouped by role:

- **Geometry resolution** controls how densely the curves are sampled.
- **Template opening** controls where each closed knot template is cut to create the open local-knot arc.
- **Embedding geometry** controls the theta shape, knot insertion intervals, and transverse radii.
- **Yamada controls** govern projection search and invariant evaluation.
- **Display controls** affect only the final visualization.

Increasing curve sampling improves geometric resolution but also makes projection/crossing detection slower. Increasing `YAMADA_NUM_ROTATION_SAMPLES` searches more projection directions and can find lower-crossing diagrams, but costs more projection work.


In [ ]:
# =========================
# Geometry resolution
# =========================
BASE_THETA_SAMPLES = 260

TEMPLATE_SAMPLES = {
    "3_1": 420,
    "4_1": 480,
    "5_1": 420,
    "7_1": 420,
}

# =========================
# Theta-graph geometry
# =========================
LEFT_VERTEX = np.array([-1.1, 0.0, 0.0])
RIGHT_VERTEX = np.array([1.1, 0.0, 0.0])

THETA_BOW_HEIGHT = 0.92
THETA_Z_WIGGLE = 0.08

# =========================
# Opening each closed knot template
# =========================
# Fractions are measured along the sampled closed template.
TEMPLATE_OPENING = {
    "3_1": dict(start_fraction=0.02, gap_fraction=0.10),
    "4_1": dict(start_fraction=0.02, gap_fraction=0.10),
    "5_1": dict(start_fraction=0.02, gap_fraction=0.10),
    "7_1": dict(start_fraction=0.02, gap_fraction=0.10),
}

# =========================
# Local-knot placement on edge e1
# =========================
SINGLE_KNOT_SLOT = (-0.47, 0.47)
DOUBLE_KNOT_SLOTS = [(-0.82, -0.08), (0.08, 0.82)]

SINGLE_KNOT_RADIUS = 0.25
DOUBLE_KNOT_RADIUS = 0.15

CONNECTOR_SAMPLES = 60
FINAL_CONNECTOR_SAMPLES = 80
UNKNOTTED_E1_SAMPLES = 320

# Edit this dictionary to add/remove cases.
VARIANTS = {
    "0_1": (),
    "3_1": ("3_1",),
    "4_1": ("4_1",),
    "5_1": ("5_1",),
    "7_1": ("7_1",),
    "3_1#3_1": ("3_1", "3_1"),
    "3_1#4_1": ("3_1", "4_1"),
    "3_1#5_1": ("3_1", "5_1"),
    "4_1#4_1": ("4_1", "4_1"),
    "4_1#5_1": ("4_1", "5_1"),
}

# =========================
# Yamada / projection controls
# =========================
YAMADA_NUM_ROTATION_SAMPLES = 18
YAMADA_ROTATION_ORDER = "ZYX"
YAMADA_NORMALIZE = True
YAMADA_N_JOBS = 1
YAMADA_METHOD = "recursive"       # alternatively: "negami"
YAMADA_CROSSING_WARNING_THRESHOLD = 12

# =========================
# Display controls
# =========================
SHOW_DISCRIMINATION_MATRIX = True
MATRIX_FIGSIZE = (7, 6)


In [ ]:
U = LEFT_VERTEX.copy()
V = RIGHT_VERTEX.copy()

def base_theta(n=BASE_THETA_SAMPLES):
    t = np.linspace(0.0, 1.0, n)
    x = U[0] + (V[0] - U[0]) * t
    curves = {
        "e1": np.c_[x, 0.0 * t, 0.0 * t],
        "e2": np.c_[
            x,
            THETA_BOW_HEIGHT * np.sin(np.pi * t),
            THETA_Z_WIGGLE * np.sin(2.0 * np.pi * t),
        ],
        "e3": np.c_[
            x,
            -THETA_BOW_HEIGHT * np.sin(np.pi * t),
            -THETA_Z_WIGGLE * np.sin(2.0 * np.pi * t),
        ],
    }

    G = nx.MultiGraph()
    G.add_node("u", pos=U.copy())
    G.add_node("v", pos=V.copy())
    for role, points in curves.items():
        G.add_edge("u", "v", role=role, pts=points)
    return G

def normalize_curve(points):
    points = np.asarray(points, dtype=float).copy()
    points -= points.mean(axis=0)
    scale = np.max(np.linalg.norm(points, axis=1))
    if scale <= 0:
        raise ValueError("Degenerate curve.")
    return points / scale

def torus_knot(p, q, n):
    t = np.linspace(0.0, 2.0 * np.pi, n, endpoint=False)
    r = 1.0 + 0.38 * np.cos(q * t)
    return normalize_curve(
        np.c_[r * np.cos(p * t), r * np.sin(p * t), 0.38 * np.sin(q * t)]
    )

def figure_eight_knot(n):
    t = np.linspace(0.0, 2.0 * np.pi, n, endpoint=False)
    return normalize_curve(
        np.c_[
            (2.0 + np.cos(2.0 * t)) * np.cos(3.0 * t),
            (2.0 + np.cos(2.0 * t)) * np.sin(3.0 * t),
            np.sin(4.0 * t),
        ]
    )

TEMPLATES = {
    "3_1": torus_knot(2, 3, TEMPLATE_SAMPLES["3_1"]),
    "4_1": figure_eight_knot(TEMPLATE_SAMPLES["4_1"]),
    "5_1": torus_knot(2, 5, TEMPLATE_SAMPLES["5_1"]),
    "7_1": torus_knot(2, 7, TEMPLATE_SAMPLES["7_1"]),
}

print("Constructed parametric knot templates:", ", ".join(TEMPLATES))


In [ ]:
def basis(a, b):
    ex = (b - a) / np.linalg.norm(b - a)
    ref = np.array([0.0, 0.0, 1.0])
    if abs(ex @ ref) > 0.9:
        ref = np.array([0.0, 1.0, 0.0])
    ey = np.cross(ref, ex)
    ey /= np.linalg.norm(ey)
    ez = np.cross(ex, ey)
    return np.vstack([ex, ey, ez])

def open_closed_template(points, *, start_fraction, gap_fraction):
    # Remove one contiguous gap from a sampled closed template.
    points = np.asarray(points, dtype=float)
    n = len(points)

    if not (0.0 <= start_fraction < 1.0):
        raise ValueError("start_fraction must satisfy 0 <= value < 1.")
    if not (0.0 < gap_fraction < 1.0):
        raise ValueError("gap_fraction must satisfy 0 < value < 1.")

    start = int(round(start_fraction * n)) % n
    gap = max(4, int(round(gap_fraction * n)))
    end = (start + gap) % n

    if start < end:
        opened = np.vstack([points[end:], points[: start + 1]])
    else:
        opened = points[end : start + 1]

    if len(opened) < 20:
        raise ValueError("Opening left too few points; decrease gap_fraction.")
    if np.linalg.norm(opened[-1] - opened[0]) < 1e-8:
        raise ValueError("Opening produced nearly coincident endpoints.")
    return opened

OPEN_TEMPLATES = {
    name: open_closed_template(points, **TEMPLATE_OPENING[name])
    for name, points in TEMPLATES.items()
}

for name, points in OPEN_TEMPLATES.items():
    print(
        f"{name:>3s}: open points={len(points):4d}, "
        f"endpoint distance={np.linalg.norm(points[-1] - points[0]):.3f}"
    )


In [ ]:
def map_arc(open_template, left, right, radius):
    left = np.asarray(left, dtype=float)
    right = np.asarray(right, dtype=float)
    mid = 0.5 * (left + right)

    a, b = open_template[0], open_template[-1]
    local = (open_template - 0.5 * (a + b)) @ basis(a, b).T

    target_length = np.linalg.norm(right - left)
    local[:, 0] *= target_length / np.linalg.norm(b - a)

    transverse = np.max(np.linalg.norm(local[:, 1:], axis=1))
    if transverse <= 0:
        raise ValueError("Template has zero transverse extent.")
    local[:, 1:] *= radius / transverse

    ex = (right - left) / target_length
    ref = np.array([0.0, 0.0, 1.0])
    if abs(ex @ ref) > 0.9:
        ref = np.array([0.0, 1.0, 0.0])

    ey = np.cross(ref, ex)
    ey /= np.linalg.norm(ey)
    ez = np.cross(ex, ey)

    points = local @ np.vstack([ex, ey, ez]) + mid
    points[0] = left
    points[-1] = right
    return points

def knotted_e1(factors):
    if not factors:
        return np.linspace(U, V, UNKNOTTED_E1_SAMPLES)

    if len(factors) == 1:
        slots = [(SINGLE_KNOT_SLOT[0], SINGLE_KNOT_SLOT[1], SINGLE_KNOT_RADIUS)]
    elif len(factors) == 2:
        slots = [
            (DOUBLE_KNOT_SLOTS[0][0], DOUBLE_KNOT_SLOTS[0][1], DOUBLE_KNOT_RADIUS),
            (DOUBLE_KNOT_SLOTS[1][0], DOUBLE_KNOT_SLOTS[1][1], DOUBLE_KNOT_RADIUS),
        ]
    else:
        raise ValueError("This controlled notebook currently supports up to two local factors.")

    pieces = []
    cursor = U.copy()

    for factor, (xa, xb, radius) in zip(factors, slots):
        left = np.array([xa, 0.0, 0.0])
        right = np.array([xb, 0.0, 0.0])
        pieces.append(np.linspace(cursor, left, CONNECTOR_SAMPLES, endpoint=False))
        pieces.append(map_arc(OPEN_TEMPLATES[factor], left, right, radius)[:-1])
        cursor = right

    pieces.append(np.linspace(cursor, V, FINAL_CONNECTOR_SAMPLES))
    points = np.vstack(pieces)
    points[0] = U
    points[-1] = V
    return points

def make_theta(factors):
    G = base_theta()
    for u, v, key, data in G.edges(keys=True, data=True):
        if data["role"] == "e1":
            G[u][v][key]["pts"] = knotted_e1(factors)
            break

    assert G.number_of_nodes() == 2
    assert G.number_of_edges() == 3
    assert sorted(dict(G.degree()).values()) == [3, 3]
    return G

EMBEDDINGS = {name: make_theta(factors) for name, factors in VARIANTS.items()}

reference_name = next(iter(EMBEDDINGS))
reference = EMBEDDINGS[reference_name]

for name, G in EMBEDDINGS.items():
    if not nx.is_isomorphic(reference, G):
        raise RuntimeError(f"{name} is not abstractly isomorphic to {reference_name}.")

print(
    f"PASS: {len(EMBEDDINGS)} embeddings share the same abstract theta graph "
    f"(V={reference.number_of_nodes()}, E={reference.number_of_edges()})."
)


## Compute KnottedGraph's Yamada polynomial

For each embedding, KnottedGraph searches sampled projection directions, selects a projection according to its projection policy, and computes the Yamada polynomial.

The number of crossings matters strongly for runtime. If this cell is slow, first experiment with:

- lowering the geometric sampling densities;
- changing `YAMADA_NUM_ROTATION_SAMPLES`;
- switching `YAMADA_METHOD` between `"recursive"` and `"negami"`;
- changing the template opening or insertion radii so the geometry admits a simpler projection.

All of those controls are in the single configuration cell above.


In [ ]:
def yamada(graph):
    result = compute_yamada_polynomial(
        graph,
        A,
        rotation_angles=None,
        rotation_order=YAMADA_ROTATION_ORDER,
        num_rotation_samples=YAMADA_NUM_ROTATION_SAMPLES,
        crossing_warning_threshold=YAMADA_CROSSING_WARNING_THRESHOLD,
        normalize=YAMADA_NORMALIZE,
        n_jobs=YAMADA_N_JOBS,
        method=YAMADA_METHOD,
        return_result=True,
    )
    return sp.expand(result.polynomial), int(result.projection.num_crossings)

def same_polynomial(a, b):
    return sp.simplify(sp.together(sp.expand(a - b))) == 0

YAMADA = {}
CROSSINGS = {}

for name, graph in EMBEDDINGS.items():
    YAMADA[name], CROSSINGS[name] = yamada(graph)
    print(f"{name:10s} crossings={CROSSINGS[name]:2d}  Yamada={YAMADA[name]}")

groups = []
unused = set(YAMADA)

while unused:
    seed = sorted(unused)[0]
    group = [
        name for name in sorted(unused)
        if same_polynomial(YAMADA[seed], YAMADA[name])
    ]
    groups.append(group)
    unused.difference_update(group)

print()
print("Total same-abstract-graph embeddings:", len(EMBEDDINGS))
print("Distinct Yamada polynomials:", len(groups))
print("Discrimination fraction:", len(groups) / len(EMBEDDINGS))
print("Yamada equivalence groups:", groups)

names = list(EMBEDDINGS)
distinguished_pairs = sum(
    1
    for i, a in enumerate(names)
    for b in names[i + 1 :]
    if not same_polynomial(YAMADA[a], YAMADA[b])
)
total_pairs = len(names) * (len(names) - 1) // 2
print(f"Pairwise distinctions: {distinguished_pairs}/{total_pairs}")

if SHOW_DISCRIMINATION_MATRIX:
    n = len(names)
    matrix = np.zeros((n, n), dtype=int)

    for i, a in enumerate(names):
        for j, b in enumerate(names):
            matrix[i, j] = int(not same_polynomial(YAMADA[a], YAMADA[b]))

    fig, ax = plt.subplots(figsize=MATRIX_FIGSIZE)
    image = ax.imshow(matrix, vmin=0, vmax=1)
    ax.set_xticks(range(n), names, rotation=60, ha="right")
    ax.set_yticks(range(n), names)
    ax.set_title("Yamada pairwise discrimination: fixed abstract theta graph")
    fig.colorbar(image, ax=ax, label="1 = distinguished by Yamada")
    plt.tight_layout()
    plt.show()


## Interpretation and hard-case extension

This experiment is deliberately self-contained inside KnottedGraph:

1. every object is verified to have the same abstract theta-graph connectivity;
2. the 3D embeddings are changed by inserting different explicit local-knot geometries;
3. KnottedGraph projects each embedding and computes its Yamada polynomial;
4. unequal polynomials certify unequal spatial embeddings.

The experiment does **not** claim that equal-polynomial cases are equivalent.

A useful harder Level-2 extension would use literature-backed coordinates for theta curves such as a Kinoshita-type example, where constituent-cycle knot types alone do not reveal the embedding topology. Such coordinates should be imported from a trusted source rather than fabricated.
